# Introduction

                     MIC PROJECT
                        │
          ┌─────────────┴─────────────┐
          │                           │
    Biotot_noncured ~800 GENERA
(preserved, never modified)       NEW 150/300 GENERA
          │                           │
          └─────────────┬─────────────┘
                        │
                    MultiTax
                        │
                        ▼
                    GTDB R232
                        │
                        ▼
                Current taxonomy 800-genus harmonised list
                        │
            ┌───────────┴───────────┐
            │                       │
    Update old reference     Classify new data
            │                       │
            ▼                       ▼
    GID identity preserved       Match / new / unknown

In [ ]:
from pathlib import Path
import pandas as pd
from multitax import GtdbTx

#gtdb = GtdbTx(version="232")
#print(gtdb)

GtdbTx(version='232', source=['https://data.gtdb.ecogenomic.org/releases/release232/232.0/ar53_taxonomy_r232.tsv.gz', 'https://data.gtdb.ecogenomic.org/releases/release232/232.0/bac120_taxonomy_r232.tsv.gz'], datetime=datetime.datetime(2026, 7, 23, 0, 26, 49, 383964))


In [ ]:
# Load  current harmonised 800-genus reference
biotot_path = Path("data/Biotot.xlsx")

biotot = pd.read_excel(biotot_path)

biotot.head()

In [11]:
# the querry will have the Genus and keep the ID column, so to not lose the ID, id already in df
genus_queries = biotot[["Genus", "GID"]].copy()
genus_queries["query"] = "g__" + genus_queries["Genus"].astype(str)
genus_queries.head()

,Genus,GID,query
0,02d06,1.0,g__02d06
1,A17,2.0,g__A17
2,Abiotrophia,3.0,g__Abiotrophia
3,Acetanaerobacterium,4.0,g__Acetanaerobacterium
4,Acetivibrio,5.0,g__Acetivibrio


In [12]:
# data from the GTDB database, to get the taxonomic ranks for each genus
#gtdb = GtdbTx(version="232")
rank_prefix = {"d__": "Kingdom", "p__": "Phylum", "c__": "Class", "o__": "Order", "f__": "Family", "g__": "Genus"}

records = []
for _, r in genus_queries.iterrows():
    row = {"GID": r["GID"], "Genus": r["Genus"], "GTDB_status": None}
    try:
        lineage_nodes = gtdb.lineage(r["query"])
        for node in lineage_nodes:
            for prefix, rankname in rank_prefix.items():
                if node.startswith(prefix):
                    row[rankname] = node[len(prefix):]
        row["GTDB_status"] = "found"
    except Exception:
        row["GTDB_status"] = "not_in_gtdb"
    records.append(row)

table_b = pd.DataFrame(records)
table_b.head()

,GID,Genus,GTDB_status,Kingdom,Phylum,Class,Order,Family
0,1.0,02d06,found,NaN,NaN,NaN,NaN,NaN
1,2.0,A17,found,NaN,NaN,NaN,NaN,NaN
2,3.0,Abiotrophia,found,Bacteria,Bacillota,Bacilli,Lactobacillales,Aerococcaceae
3,4.0,Acetanaerobacterium,found,Bacteria,Bacillota,Clostridia,Oscillospirales,Ruminococcaceae
4,5.0,Acetivibrio,found,Bacteria,Bacillota,Clostridia,Acetivibrionales,Acetivibrionaceae


# Query GTDB R232 with MultiTax

Create a gtdb_taxonomy dataframe.
Query data from the GTDB database, to get the taxonomic ranks for each genus

In [13]:
# Query data from the GTDB database, to get the taxonomic ranks for each genus
def strip_unclassified(genus_name):
    """Acetivibrio_unclassified -> Acetivibrio. Returns None if no suffix to strip."""
    for suffix in ("_unclassified", "_uncultured"):
        if genus_name.endswith(suffix):
            return genus_name[: -len(suffix)]
    return None


rank_prefix = {"d__": "Kingdom", "p__": "Phylum", "c__": "Class", "o__": "Order", "f__": "Family", "g__": "Genus"}

def lookup_lineage(query_genus):
    """Returns a dict of rank->value if a real lineage was found, else None."""
    try:
        lineage_nodes = gtdb.lineage(f"g__{query_genus}")
    except Exception:
        return None
    parsed = {}
    for node in lineage_nodes:
        for prefix, rankname in rank_prefix.items():
            if node.startswith(prefix):
                parsed[rankname] = node[len(prefix):]
    # a real hit must contain at least Phylum -- an empty/near-empty result means no match
    if "Phylum" not in parsed:
        return None
    return parsed


records = []
for _, r in genus_queries.iterrows():
    original_genus = r["Genus"]
    row = {"GID": r["GID"], "Genus": original_genus}

    parsed = lookup_lineage(original_genus)
    match_type = "direct"

    if parsed is None:
        stripped = strip_unclassified(original_genus)
        if stripped:
            parsed = lookup_lineage(stripped)
            match_type = f"resolved_via_base_genus({stripped})" if parsed else "not_in_gtdb"
        else:
            match_type = "not_in_gtdb"

    if parsed:
        row.update(parsed)
        row["Genus"] = original_genus  # keep your original name, e.g. Acetivibrio_unclassified, not overwritten
    row["GTDB_status"] = match_type
    records.append(row)

table_b = pd.DataFrame(records)
table_b.head()

AttributeError: 'float' object has no attribute 'endswith'

In [ ]:
merged = biotot.merge(table_b, on="Genus", how="left", suffixes=("_yours", "_gtdb"))

def make_comment(row):
    if row["GTDB_status"] == "skipped_placeholder":
        return "placeholder/composite name, not checked against GTDB"
    if row["GTDB_status"] == "not_in_gtdb":
        return "genus not found in GTDB (may be NCBI-only, or GTDB uses a different name)"
    diffs = []
    for rank in ["Phylum", "Class", "Order", "Family"]:
        yours = row.get(f"{rank}_yours", row.get(rank))
        gtdb_val = row.get(f"{rank}_gtdb")
        if pd.notna(gtdb_val) and yours != gtdb_val:
            diffs.append(f"{rank}: '{yours}' -> '{gtdb_val}'")
    return "; ".join(diffs) if diffs else "matches GTDB"

merged["Comment"] = merged.apply(make_comment, axis=1)
merged[["Genus", "Comment"]].to_excel("data/Biotot_vs_GTDB_comparison.xlsx", index=False)
merged[merged["Comment"].str.startswith(("Phylum", "Class", "Order", "Family"))]